In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# 오류 완화 구성

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



> **Note:** 새로운 실행 모델의 베타 릴리스가 이제 제공됩니다. 지시된 실행 모델은 오류 완화 워크플로를 사용자 정의할 때 더 많은 유연성을 제공합니다. 자세한 내용은 [지시된 실행 모델](/guides/directed-execution-model) 가이드를 참고하세요.


<details>
<summary><b>패키지 버전</b></summary>

이 페이지의 코드는 다음 요구 사항을 사용하여 개발되었습니다.
이 버전 또는 더 최신 버전을 사용하는 것을 권장합니다.

```
qiskit-ibm-runtime~=0.43.1
```
</details>
오류 완화 기법은 실행 시점에 장치의 노이즈를 모델링하여
사용자가 회로 오류를 완화할 수 있도록 합니다. 이는 일반적으로
모델 학습과 관련된 양자 전처리 오버헤드와, 생성된 모델을
사용하여 원시 결과의 오류를 완화하기 위한 고전적 후처리
오버헤드를 초래합니다.

Estimator 프리미티브는 [TREX](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#measure_mitigation), [ZNE](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#zne), [PEC](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#pec), [PEA](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options)를 포함한 여러 오류 완화 기법을 지원합니다. 각각에 대한 설명은 [오류 완화 및 억제 기법](error-mitigation-and-suppression-techniques)을 참고하세요. 프리미티브를 사용할 때, 개별 방법을 켜거나 끌 수 있습니다. 자세한 내용은 [사용자 정의 오류 설정](#advanced-error) 섹션을 참고하세요.

> **Note:** Sampler는 오류 완화를 지원하지 않지만, [mthree](https://qiskit.github.io/qiskit-addon-mthree/)(행렬 없는 측정 완화) 패키지를 사용하여 로컬에서 오류 완화를 수행할 수 있습니다.

Estimator는 또한 `resilience_level`을 지원합니다. 복원력 수준은 오류에 대해
얼마나 많은 복원력을 구축할지 지정합니다. 높은 수준은 더 정확한 결과를 생성하지만,
처리 시간이 길어집니다. 복원력 수준은 프리미티브 쿼리에 오류 완화를 적용할 때
비용/정확도 트레이드오프를 구성하는 데 사용할 수 있습니다. 오류 완화는 관련 회로의
컬렉션 또는 앙상블의 출력을 처리하여 결과의 오류(편향)를 줄입니다. 오류 감소의
정도는 적용된 방법에 따라 달라집니다. 복원력 수준은 사용자가 애플리케이션에
적합한 비용/정확도 트레이드를 추론할 수 있도록 오류 완화 방법에 대한
세부적인 선택을 추상화합니다.

이를 감안할 때, 각 수준은 서로 다른 시간-정확도 트레이드오프를
실험할 수 있도록 점점 더 큰 양자 샘플링 오버헤드를 가진
방법 또는 방법들에 해당합니다. 다음 표는 각 프리미티브에
대해 어떤 수준과 해당 방법이 사용 가능한지
보여줍니다.

> **Info:** 오류 완화는 작업에 따라 다르므로, 적용 가능한 기법은
> 분포를 샘플링하는지 또는 기댓값을 생성하는지에 따라
> 달라집니다.

<span id="resilience-table"></span>

Estimator는 다음의 복원력 수준을 지원합니다. Sampler는 복원력 수준을 지원하지 않습니다.

| 복원력 수준       | 정의                                                                                                  | 기법                                                                      |
|------------------|-------------------------------------------------------------------------------------------------------|---------------------------------------------------------------------------|
| 0                | 완화 없음                                                                                              | 없음                                                                      |
| 1 [기본값]        | 최소 완화 비용: 판독 오류와 관련된 오류 완화                                                          | Twirled Readout Error eXtinction (TREX) 측정 트월링                       |
| 2                | 중간 완화 비용. 일반적으로 추정량의 편향을 줄이지만, 편향이 0임을 보장하지는 않습니다.                 | Level 1 + Zero Noise Extrapolation (ZNE) 및 게이트 트월링

> **Info:** 복원력 수준은 현재 베타 상태이므로 샘플링 오버헤드와
> 솔루션 품질은 회로마다 다를 수 있습니다. 새로운 기능,
> 고급 옵션, 관리 도구는 순차적으로 릴리스됩니다.
> 특정 오류 완화 방법이 각 복원력 수준에서 적용된다는
> 보장은 없습니다.

## 복원력 수준으로 Estimator 구성
복원력 수준을 사용하여 오류 완화 기법을 지정하거나, [사용자 정의 오류 설정](#advanced-error)에 설명된 대로 개별적으로 사용자 정의 기법을 설정할 수 있습니다.

<details>
<summary>복원력 수준 0</summary>

사용자 프로그램에 오류 완화가 적용되지 않습니다.

</details>

<details>
<summary>복원력 수준 1</summary>

Level 1은 Twirled Readout Error eXtinction(TREX)으로 알려진 모델 프리 기법을 적용하여 **판독 오류 완화**와 **측정 트월링**을
적용합니다. 측정 직전에 X 게이트를 통해 qubit을 무작위로
뒤집음으로써 측정과 관련된 노이즈 채널을 대각화하여
측정 오류를 줄입니다. 대각 노이즈 채널의 재조정 항은
영상태로 초기화된 무작위 회로를 벤치마킹함으로써
학습됩니다. 이를 통해 서비스는 판독 노이즈에서 비롯된
기댓값의 편향을 제거할 수 있습니다. 이 접근 방식은 [Model-free
readout-error mitigation for quantum expectation
values](https://arxiv.org/abs/2012.09738)에서 더 자세히 설명됩니다.

</details>

<details>
<summary>복원력 수준 2</summary>

Level 2는 **Level 1에 포함된 오류 완화 기법**을 적용하고 **게이트 트월링**도 적용하며 **Zero Noise Extrapolation 방법(ZNE)**을 사용합니다. ZNE는 다양한
노이즈 계수에 대해 관측 가능량의 기댓값을 계산하고
(증폭 단계), 그런 다음 측정된 기댓값을 사용하여
노이즈 없는 극한에서 이상적인 기댓값을 추론합니다
(외삽 단계). 이 접근 방식은 기댓값의 오류를 줄이는
경향이 있지만, 편향되지 않은 결과를 생성한다는
보장은 없습니다.

![이 이미지는 그래프를 보여줍니다. x축은 노이즈 증폭 계수로 표시됩니다. y축은 기댓값으로 표시됩니다. 상승하는 선은 완화된 값으로 표시됩니다. 선 근처의 점들은 노이즈 증폭 값입니다. X축 바로 위에 정확한 값으로 표시된 수평선이 있습니다.](../docs/images/guides/configure-error-mitigation/resilience-2.svg "ZNE 방법의 예시")

이 방법의 오버헤드는 노이즈 계수의 수에 따라 확장됩니다.
기본 설정은 세 가지 노이즈 계수에서 기댓값을 샘플링하며,
이 복원력 수준을 사용할 때 약 3배의 오버헤드가 발생합니다.

Level 2에서는 TREX 방법이 측정 직전에 X 게이트를 통해 qubit을 무작위로 뒤집고,
X 게이트가 적용된 경우 해당 측정 비트를 뒤집습니다. 이 접근 방식은 [Model-free
readout-error mitigation for quantum expectation
values](https://arxiv.org/abs/2012.09738)에서 더 자세히 설명됩니다.

</details>

### 예제
`EstimatorV2` 인터페이스를 통해 사용자는 관측 가능량의 기댓값에서
오류를 줄이기 위해 다양한 오류 완화 방법과 원활하게
작업할 수 있습니다. 다음 코드는 단순히 `resilience_level 2`를 설정하여
Zero Noise Extrapolation과 판독 오류 완화를
사용합니다.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(backend, options={"resilience_level": 2})

<span id="advanced-error"></span>
## 사용자 정의 오류 설정
동적 디커플링, 게이트 및 측정 트월링, 측정 오류 완화, PEC, ZNE를 포함한 개별 오류 완화 및 억제 방법을 켜거나 끌 수 있습니다. 각각에 대한 설명은 [오류 완화 및 억제 기법](error-mitigation-and-suppression-techniques)을 참고하세요.

> **Note:** - 모든 옵션이 두 프리미티브 모두에 사용 가능한 것은 아닙니다. 사용 가능한 옵션 목록은 [사용 가능한 옵션 표](runtime-options-overview#options-table)를 참고하세요.
> - 모든 방법이 모든 유형의 회로에서 함께 작동하는 것은 아닙니다. 자세한 내용은 [기능 호환성 표](runtime-options-overview#options-compatibility-table)를 참고하세요.

<Tabs>
  <TabItem value="Estimator" label="Estimator">
    ```python
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(backend)
options = estimator.options
# Turn on gate twirling.
options.twirling.enable_gates = True
# Turn on measurement error mitigation.
options.resilience.measure_mitigation = True

print(f">>> gate twirling is turned on: {estimator.options.twirling.enable_gates}")
print(f">>> measurement error mitigation is turned on: {estimator.options.resilience.measure_mitigation}")
```
  </TabItem>

  <TabItem value="Sampler" label="Sampler">